# MILO — verificación del núcleo, Sprint 1
Este notebook ejecuta cálculos y validaciones reales offline. **No es todavía la demo académica del agente Gemini/LangGraph**. Las cifras proceden de USDA y las ausencias se conservan. Ejecutar después de instalar el paquete y preparar el catálogo.

In [1]:
from pathlib import Path
import json
from milo.models import Recipe, RecipeIngredient, UserProfile, DietaryRestriction
from milo.repositories.catalog import FoodCatalog
from milo.domain.nutrition import calculate_nutrition, calculate_recipe_nutrition
from milo.domain.constraints import validate_recipe
from milo.domain.recipes import adjust_portions
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
catalog = FoodCatalog.from_csv(root / "data/structured/foods.csv")
recipes = [Recipe.model_validate(r) for r in json.loads((root / "data/structured/recipes.json").read_text(encoding="utf-8"))]
print(f"{len(catalog.foods)} alimentos; {len(recipes)} recetas")

70 alimentos; 12 recetas


## 1. Composición y procedencia verificables

In [2]:
food = catalog.lookup_food("fdc_2646170")
print(food.name_es, food.fdc_id, food.source_id, food.source_version)
print(food.nutrients_per_100g.rounded())

Pechuga de pollo sin piel ni hueso, cruda 2646170 USDA_FOUNDATION_2026_04 2026-04-30
{'kcal': '112.20', 'protein_g': '22.53', 'carbs_g': '0.00', 'fat_g': '1.93', 'fiber_g': None}


## 2. Escalado de 150 g con Python

In [3]:
portion = RecipeIngredient(ingredient_name=food.name_es, food_id=food.food_id, grams=150, state="raw")
result = calculate_nutrition([portion], catalog)
print(result.totals.rounded(), "macros completos:", result.complete, "aproximado:", result.approximate)
assert result.totals.kcal == food.nutrients_per_100g.kcal * 150 / 100

{'kcal': '168.30', 'protein_g': '33.79', 'carbs_g': '0.00', 'fat_g': '2.90', 'fiber_g': None} macros completos: True aproximado: True


## 3. Receta y raciones; sin aritmética del LLM

In [4]:
recipe = recipes[0]
result = calculate_recipe_nutrition(recipe, catalog)
print(recipe.title, recipe.review_status)
print(result.per_serving.rounded())
assert validate_recipe(recipe, catalog).valid

Bowl de pollo y arroz verified
{'kcal': '586.38', 'protein_g': '42.51', 'carbs_g': '68.86', 'fat_g': '14.43', 'fiber_g': None}


## 4. Ajuste uniforme de cantidades a 700 kcal por ración

In [5]:
adjusted = adjust_portions(recipe, 700, catalog)
print([(i.ingredient_name, str(i.grams)) for i in adjusted.ingredients])
print(calculate_recipe_nutrition(adjusted, catalog).per_serving.rounded())

[('Pechuga de pollo sin piel ni hueso, cruda', '191.0028380212965016257683213'), ('Arroz blanco de grano largo sin enriquecer, seco', '95.50141901064825081288416064'), ('Tomate roma, crudo', '143.2521285159723762193262410'), ('Aceite de oliva para ensalada o cocina (genérico USDA SR)', '11.93767737633103135161052008')]
{'kcal': '700.00', 'protein_g': '50.74', 'carbs_g': '82.20', 'fat_g': '17.23', 'fiber_g': None}


## 5. Rechazo de conflicto con alergia conocida

In [6]:
profile = UserProfile(user_id="notebook-demo", restrictions=[DietaryRestriction(kind="allergy", value="egg")])
validation = validate_recipe(recipes[1], catalog, profile)
print(validation.model_dump())
assert not validation.valid

{'valid': False, 'errors': ['allergen_conflict:fdc_748967:egg'], 'warnings': ['Alérgenos deducidos del alimento; no acredita ausencia de trazas ni contaminación cruzada.'], 'allergens': ['egg']}


## 6. Un dato ausente sigue siendo null

In [7]:
oil = catalog.lookup_food("fdc_748608")
unknown = calculate_nutrition([RecipeIngredient(ingredient_name=oil.name_es, food_id=oil.food_id, grams=10, state=oil.state)], catalog)
print(unknown.model_dump(mode="json"))
assert not unknown.complete and unknown.totals.kcal is None

{'totals': {'kcal': None, 'protein_g': None, 'carbs_g': None, 'fat_g': None, 'fiber_g': None}, 'per_serving': {'kcal': None, 'protein_g': None, 'carbs_g': None, 'fat_g': None, 'fiber_g': None}, 'servings': 1, 'complete': False, 'approximate': True, 'missing': {'0:Aceite de oliva virgen extra': ['kcal', 'protein_g', 'carbs_g', 'fat_g', 'fiber_g']}, 'issues': [], 'food_ids': ['fdc_748608'], 'basis': 'Suma de pesos comestibles de entrada; sin factores de retención o rendimiento de cocción.'}


## 7. Recetas pendientes y fuentes documentales

In [8]:
report = json.loads((root / "evaluation/recipe_report.json").read_text(encoding="utf-8"))
print({k: report[k] for k in ["count", "verified", "pending"]})
for entry in report["recipes"]:
    if not entry["valid"]:
        print(entry["recipe_id"], entry["errors"])
print(json.loads((root / "evaluation/ingestion_report.json").read_text(encoding="utf-8")))

{'count': 12, 'verified': 7, 'pending': 5}
r004 ['allergens_pending:fdc_2758998', 'allergens_pending:fdc_334194', 'allergens_pending:fdc_2685581']
r005 ['allergens_pending:fdc_2644288']
r007 ['allergens_pending:fdc_2758994']
r009 ['allergens_pending:fdc_2758996']
r011 ['allergens_pending:fdc_172475']
{'sources': [{'source_id': 'AESAN_2022', 'pages': 19, 'empty_pages': [], 'chunks': 129, 'words': 4378, 'ocr_needed': False, 'section_method': 'heuristic_headings_with_page_boundaries'}, {'source_id': 'USDA_FIELDS', 'pages': 17, 'empty_pages': [], 'chunks': 20, 'words': 3706, 'ocr_needed': False, 'section_method': 'heuristic_headings_with_page_boundaries'}], 'domain_chunks': 129, 'academic_corpus_ready': False, 'reason': 'AESAN es una guía de 19 páginas; el diccionario USDA no sustituye conocimiento culinario. Ampliar corpus y revisar reutilización.', 'embeddings_executed': False}


Pendiente para la entrega académica: agente LangGraph, memoria, embeddings reales, Chroma y cinco consultas conversacionales con respuestas y fuentes. No se presentan como resultados los pasos todavía no ejecutados.